# ACF -- Baseline 1 training on Colab (GPU)

Trains the plain single-modality 3D U-Net (`src/models/baselines.py`) on Task06_Lung via `src/training/train.py`. Steps in `docs/RUNBOOK_colab.md`.

**Runtime -> Change runtime type -> a GPU** before starting. Run the cells top to bottom. The launch cell (section 7) is safe to re-run after a disconnect: with `--resume` it continues from the newest readable checkpoint on Drive, and starts fresh if there is none.

Prerequisites (one-off): the repo is committed and pushed to a remote you can clone, and `Task06_Lung` is uploaded to `MyDrive/hecktor_major_project/data/Task06_Lung`.

## 1. Settings -- the only cell you should need to edit

In [ ]:
REPO_URL   = ''   # e.g. https://github.com/<you>/<repo>.git  (private repo: see runbook)
BRANCH     = 'main'
CONFIG     = 'configs/train_baseline_decathlon_lung.yaml'
RUN_NAME   = 'baseline1_decathlon_lung_seed42'   # names the Drive checkpoint folder and the wandb run
USE_WANDB  = False
EXTRA_OVERRIDES = []   # e.g. ['train.n_epochs=2'] for a timing check; each is dotted.key=value

DRIVE_ROOT = '/content/drive/MyDrive/hecktor_major_project'

## 2. Mount Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir(DRIVE_ROOT), f'{DRIVE_ROOT} not found -- create it in Drive and upload the data first'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints/{RUN_NAME}'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)
print('existing:', sorted(os.listdir(CKPT_DIR)))

## 3. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), 'no GPU -- Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 4. Clone the repo and install dependencies

In [ ]:
assert REPO_URL, 'set REPO_URL in section 1'
REPO_DIR = '/content/hecktor_major_project'
if not os.path.isdir(REPO_DIR):
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" fetch origin && git -C "$REPO_DIR" checkout "$BRANCH" && git -C "$REPO_DIR" pull --ff-only
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
!git rev-parse HEAD

## 5. Data: copy from Drive to the VM's local disk

Reading hundreds of MB of `.nii.gz` per epoch straight off the Drive mount is slow, so copy once per session to local disk and symlink it in. The dataset is re-read and re-resampled every epoch (no caching), so local disk matters. `._*` files are macOS metadata junk and are skipped.

In [ ]:
drive_data, local_data = f'{DRIVE_ROOT}/data/Task06_Lung', '/content/data/Task06_Lung'
assert os.path.isdir(drive_data), f'upload Task06_Lung to {drive_data}'
if not os.path.exists(f'{local_data}/dataset.json'):
    os.makedirs('/content/data', exist_ok=True)
    !rsync -a --exclude '._*' "$drive_data/" "$local_data/"
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/Task06_Lung'):
    os.symlink(local_data, 'data/Task06_Lung')
!ls data/Task06_Lung && ls data/Task06_Lung/imagesTr | wc -l

## 6. (Optional) wandb login -- only if `USE_WANDB = True`

In [ ]:
if USE_WANDB:
    import wandb
    wandb.login()

## 7. Launch or resume

Identical command for a first launch and for resuming after a dead session. The Drive path and run name are passed as overrides, so no generated config file is written and the git checkout stays clean (the commit hash logged with the run means what it says). The overrides are part of the config logged to wandb and stored in every checkpoint.

In [ ]:
overrides = [f'checkpoint.dir={CKPT_DIR}', f'wandb.run_name={RUN_NAME}', f'wandb.enabled={str(USE_WANDB).lower()}', *EXTRA_OVERRIDES]
OVR = ' '.join(f'--override "{o}"' for o in overrides)
!python -u -m src.training.train --config $CONFIG --resume $OVR

## 8. Check what's on Drive

Newest `epoch_*.pt` files (last 3 are kept) plus `best.pt`. Everything under `checkpoints/<RUN_NAME>/` is what to download for results; see the runbook.

In [ ]:
!ls -la --time-style=long-iso "$CKPT_DIR"